In [1]:
import os, time
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

In [2]:
COMBINED_DIR       = "./combined_dataset/train"
REALWORLD_TEST_DIR = "./realworld_test"

SELECTED_CLASSES = [
    "Tomato___Early_blight",
    "Tomato___Late_blight",
    "Tomato___healthy",
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "Tomato___Leaf_Mold",
]

IMG_SIZE      = 224
BATCH_SIZE    = 32
PHASE1_EPOCHS  = 5
PHASE2_EPOCHS  = 20
NUM_CLASSES   = len(SELECTED_CLASSES)
RANDOM_SEED   = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device     : {DEVICE}")
print(f"Classes    : {NUM_CLASSES}")
print(f"Class list : {SELECTED_CLASSES}")

Device     : cpu
Classes    : 5
Class list : ['Tomato___Early_blight', 'Tomato___Late_blight', 'Tomato___healthy', 'Tomato___Tomato_Yellow_Leaf_Curl_Virus', 'Tomato___Leaf_Mold']


In [3]:
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

eval_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

In [4]:
full_dataset = datasets.ImageFolder(root=COMBINED_DIR, transform=train_transforms)

selected_idx = [full_dataset.class_to_idx[c] for c in SELECTED_CLASSES]
label_remap  = {old: new for new, old in enumerate(selected_idx)}

filtered_samples = [
    (path, label_remap[label])
    for path, label in full_dataset.samples
    if label in selected_idx
]

full_dataset.samples      = filtered_samples
full_dataset.targets      = [l for _, l in filtered_samples]
full_dataset.classes      = SELECTED_CLASSES
full_dataset.class_to_idx = {c: i for i, c in enumerate(SELECTED_CLASSES)}

print(f"Total images: {len(full_dataset)}")
for cls in SELECTED_CLASSES:
    idx   = full_dataset.class_to_idx[cls]
    count = sum(1 for _, l in full_dataset.samples if l == idx)
    print(f"  {cls:<52} {count:>5} images")

Total images: 11341
  Tomato___Early_blight                                 1079 images
  Tomato___Late_blight                                  2010 images
  Tomato___healthy                                      1635 images
  Tomato___Tomato_Yellow_Leaf_Curl_Virus                5580 images
  Tomato___Leaf_Mold                                    1037 images


In [5]:
all_indices = list(range(len(full_dataset)))
all_targets = full_dataset.targets

train_idx, temp_idx = train_test_split(
    all_indices, test_size=0.30,
    stratify=all_targets, random_state=RANDOM_SEED
)
temp_targets = [all_targets[i] for i in temp_idx]
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.50,
    stratify=temp_targets, random_state=RANDOM_SEED
)

eval_dataset = datasets.ImageFolder(root=COMBINED_DIR, transform=eval_transforms)
eval_dataset.samples      = filtered_samples
eval_dataset.targets      = [l for _, l in filtered_samples]
eval_dataset.classes      = SELECTED_CLASSES
eval_dataset.class_to_idx = {c: i for i, c in enumerate(SELECTED_CLASSES)}

train_dataset = Subset(full_dataset, train_idx)
val_dataset   = Subset(eval_dataset, val_idx)
test_dataset  = Subset(eval_dataset, test_idx)

print(f"Train : {len(train_dataset):>5} images")
print(f"Val   : {len(val_dataset):>5} images")
print(f"Test  : {len(test_dataset):>5} images")

Train :  7938 images
Val   :  1701 images
Test  :  1702 images


In [6]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=0, pin_memory=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=0, pin_memory=False)

print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")
print(f"Test batches  : {len(test_loader)}")

Train batches : 249
Val batches   : 54
Test batches  : 54


In [7]:
label_counts  = Counter(full_dataset.targets)
total_samples = sum(label_counts.values())

class_weights = torch.tensor([
    total_samples / (NUM_CLASSES * label_counts[i])
    for i in range(NUM_CLASSES)
], dtype=torch.float).to(DEVICE)

print("Class weights:")
for i, cls in enumerate(SELECTED_CLASSES):
    short = cls.replace("Tomato___","").replace("_"," ")
    print(f"  {short:<40} {class_weights[i].item():.4f}")

Class weights:
  Early blight                             2.1021
  Late blight                              1.1285
  healthy                                  1.3873
  Tomato Yellow Leaf Curl Virus            0.4065
  Leaf Mold                                2.1873


In [8]:
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

in_features = model.fc.in_features
model.fc = nn.Linear(in_features, NUM_CLASSES)

model = model.to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,}")
print(f"FC layer             : {in_features} → {NUM_CLASSES}")

Total parameters     : 23,518,277
Trainable parameters : 23,518,277
FC layer             : 2048 → 5


In [9]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for batch_idx, (images, labels) in enumerate(loader):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted  = outputs.max(1)
        total        += labels.size(0)
        correct      += predicted.eq(labels).sum().item()

        if (batch_idx + 1) % 50 == 0:
            print(f"    Batch {batch_idx+1}/{len(loader)} — "
                  f"loss: {running_loss/total:.4f} — "
                  f"acc: {100.*correct/total:.1f}%", flush=True)

    return running_loss / total, 100. * correct / total


def validate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss    = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            _, predicted  = outputs.max(1)
            total        += labels.size(0)
            correct      += predicted.eq(labels).sum().item()

    return running_loss / total, 100. * correct / total

print("Helper functions defined.")

Helper functions defined.


In [10]:
# PHASE 1: Train classifier head only (backbone frozen)
print("=" * 60)
print("PHASE 1 — Head only training (backbone frozen)")
print("=" * 60)

# Freeze ALL layers
for param in model.parameters():
    param.requires_grad = False

# Unfreeze only the new FC head
for param in model.fc.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters in Phase 1: {trainable:,}  (head only)")

criterion  = nn.CrossEntropyLoss(weight=class_weights)
optimizer1 = optim.Adam(model.fc.parameters(), lr=1e-3)

history_p1 = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

header = "Epoch | Train Loss | Train Acc | Val Loss |  Val Acc |   Time"
print(header)
print("-" * 60)

for epoch in range(PHASE1_EPOCHS):
    start = time.time()
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer1, criterion, DEVICE)
    val_loss,   val_acc   = validate(model, val_loader, criterion, DEVICE)

    history_p1["train_loss"].append(train_loss)
    history_p1["train_acc"].append(train_acc)
    history_p1["val_loss"].append(val_loss)
    history_p1["val_acc"].append(val_acc)

    elapsed = time.time() - start
    epoch_num = epoch + 1
    print(f"{epoch_num:>6} | {train_loss:>10.4f} | {train_acc:>8.2f}% | "
          f"{val_loss:>8.4f} | {val_acc:>7.2f}% | {elapsed:>5.1f}s")

print(f"Phase 1 complete.")

PHASE 1 — Head only training (backbone frozen)
Trainable parameters in Phase 1: 10,245  (head only)
Epoch | Train Loss | Train Acc | Val Loss |  Val Acc |   Time
------------------------------------------------------------
    Batch 50/249 — loss: 1.2609 — acc: 58.4%
    Batch 100/249 — loss: 1.0051 — acc: 70.1%
    Batch 150/249 — loss: 0.8749 — acc: 75.0%
    Batch 200/249 — loss: 0.7859 — acc: 77.9%
     1 |     0.7307 |    79.59% |   0.4476 |   87.24% | 3134.4s
    Batch 50/249 — loss: 0.4831 — acc: 85.9%
    Batch 100/249 — loss: 0.4611 — acc: 87.2%
    Batch 150/249 — loss: 0.4425 — acc: 87.8%
    Batch 200/249 — loss: 0.4228 — acc: 88.5%
     2 |     0.4154 |    88.61% |   0.3679 |   89.89% | 2684.3s
    Batch 50/249 — loss: 0.3785 — acc: 88.6%
    Batch 100/249 — loss: 0.3860 — acc: 88.7%
    Batch 150/249 — loss: 0.3790 — acc: 89.2%
    Batch 200/249 — loss: 0.3749 — acc: 89.6%
     3 |     0.3624 |    90.09% |   0.3073 |   91.95% | 1118.5s
    Batch 50/249 — loss: 0.3250 — ac

In [11]:
# PHASE 2: Unfreeze layer3 + layer4, fine-tune jointly
print("=" * 60)
print("PHASE 2 — Fine-tuning backbone (layer3 + layer4 + head)")
print("=" * 60)

# Unfreeze last two residual blocks
for param in model.layer3.parameters():
    param.requires_grad = True
for param in model.layer4.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters in Phase 2: {trainable:,}")

# Different learning rates per parameter group:
#   backbone layers → very low LR (nudge, not overwrite)
#   classifier head → higher LR  (still adapting)
optimizer2 = optim.Adam([
    {"params": model.layer3.parameters(), "lr": 1e-5},
    {"params": model.layer4.parameters(), "lr": 1e-5},
    {"params": model.fc.parameters(),     "lr": 1e-4},
], weight_decay=1e-4)

scheduler2 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer2, mode="max", factor=0.5, patience=4
)

history_p2 = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_acc = 0.0
best_epoch   = 0

header = "Epoch | Train Loss | Train Acc | Val Loss |  Val Acc |   Time"
print(header)
print("-" * 60)

for epoch in range(PHASE2_EPOCHS):
    start = time.time()
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer2, criterion, DEVICE)
    val_loss,   val_acc   = validate(model, val_loader, criterion, DEVICE)
    scheduler2.step(val_acc)

    history_p2["train_loss"].append(train_loss)
    history_p2["train_acc"].append(train_acc)
    history_p2["val_loss"].append(val_loss)
    history_p2["val_acc"].append(val_acc)

    elapsed = time.time() - start
    epoch_num = epoch + 1
    print(f"{epoch_num:>6} | {train_loss:>10.4f} | {train_acc:>8.2f}% | "
          f"{val_loss:>8.4f} | {val_acc:>7.2f}% | {elapsed:>5.1f}s")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch   = epoch + 1
        torch.save(model.state_dict(), "finetuned_resnet50_best.pth")
        print(f"         ✓ Best model saved (val acc: {best_val_acc:.2f}%)")

print(f"Phase 2 complete. Best val acc: {best_val_acc:.2f}% at epoch {best_epoch}")

PHASE 2 — Fine-tuning backbone (layer3 + layer4 + head)
Trainable parameters in Phase 2: 22,073,349
Epoch | Train Loss | Train Acc | Val Loss |  Val Acc |   Time
------------------------------------------------------------
    Batch 50/249 — loss: 0.2426 — acc: 92.7%
    Batch 100/249 — loss: 0.2509 — acc: 93.0%
    Batch 150/249 — loss: 0.2247 — acc: 93.7%
    Batch 200/249 — loss: 0.2136 — acc: 94.1%
     1 |     0.2065 |    94.18% |   0.1541 |   95.83% | 2043.5s
         ✓ Best model saved (val acc: 95.83%)
    Batch 50/249 — loss: 0.1144 — acc: 96.4%
    Batch 100/249 — loss: 0.1268 — acc: 96.4%
    Batch 150/249 — loss: 0.1368 — acc: 96.3%
    Batch 200/249 — loss: 0.1349 — acc: 96.4%
     2 |     0.1330 |    96.38% |   0.1206 |   96.83% | 2602.3s
         ✓ Best model saved (val acc: 96.83%)
    Batch 50/249 — loss: 0.1116 — acc: 96.8%
    Batch 100/249 — loss: 0.1065 — acc: 96.8%
    Batch 150/249 — loss: 0.1028 — acc: 97.0%
    Batch 200/249 — loss: 0.1084 — acc: 96.8%
     3 |

In [12]:
train_loss_full = history_p1["train_loss"] + history_p2["train_loss"]
val_loss_full   = history_p1["val_loss"]   + history_p2["val_loss"]
train_acc_full  = history_p1["train_acc"]  + history_p2["train_acc"]
val_acc_full    = history_p1["val_acc"]    + history_p2["val_acc"]
total_epochs    = PHASE1_EPOCHS + PHASE2_EPOCHS
epochs_range    = range(1, total_epochs + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs_range, train_loss_full, label="Train", color="#3B8BD4", linewidth=2)
ax1.plot(epochs_range, val_loss_full,   label="Val",   color="#D85A30", linewidth=2)
ax1.axvline(x=PHASE1_EPOCHS, color="gray", linestyle="--", alpha=0.7, label="Phase 1→2")
ax1.set_title("Fine-Tuned ResNet50 — Loss", fontsize=13, fontweight="bold")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(epochs_range, train_acc_full, label="Train", color="#3B8BD4", linewidth=2)
ax2.plot(epochs_range, val_acc_full,   label="Val",   color="#D85A30", linewidth=2)
ax2.axvline(x=PHASE1_EPOCHS, color="gray", linestyle="--", alpha=0.7, label="Phase 1→2")
ax2.set_title("Fine-Tuned ResNet50 — Accuracy", fontsize=13, fontweight="bold")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy (%)")
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("finetuned_training_curves.png", dpi=150)
plt.close()

In [13]:
model.load_state_dict(torch.load("finetuned_resnet50_best.pth", map_location=DEVICE))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images  = images.to(DEVICE)
        outputs = model(images)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
short_names = [c.replace("Tomato___","").replace("_"," ") for c in SELECTED_CLASSES]

test_acc = 100. * (all_preds == all_labels).sum() / len(all_labels)
print(f"Fine-Tuned ResNet50 — Standard Test Accuracy: {test_acc:.2f}%")
print()
print(classification_report(all_labels, all_preds,
                             target_names=short_names, digits=4))

Fine-Tuned ResNet50 — Standard Test Accuracy: 98.88%

                               precision    recall  f1-score   support

                 Early blight     0.9753    0.9753    0.9753       162
                  Late blight     0.9739    0.9868    0.9803       302
                      healthy     0.9879    0.9919    0.9899       246
Tomato Yellow Leaf Curl Virus     0.9964    0.9976    0.9970       837
                    Leaf Mold     0.9933    0.9548    0.9737       155

                     accuracy                         0.9888      1702
                    macro avg     0.9853    0.9813    0.9832      1702
                 weighted avg     0.9889    0.9888    0.9888      1702



In [14]:
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=short_names, yticklabels=short_names,
            linewidths=0.5, annot_kws={"size": 11})
plt.title("Fine-Tuned ResNet50 — Confusion Matrix (Standard Test)",
          fontsize=13, fontweight="bold")
plt.xlabel("Predicted"); plt.ylabel("True")
plt.xticks(rotation=30, ha="right"); plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig("finetuned_confusion_matrix_standard.png", dpi=150)
plt.close()

In [15]:
if not os.path.exists(REALWORLD_TEST_DIR):
    print(f"Real-world dir not found: {REALWORLD_TEST_DIR}")
else:
    rw_dataset = datasets.ImageFolder(root=REALWORLD_TEST_DIR,
                                       transform=eval_transforms)
    rw_dataset.class_to_idx = {c: i for i, c in enumerate(SELECTED_CLASSES)}
    rw_dataset.classes       = SELECTED_CLASSES
    rw_dataset.samples = [
        (p, rw_dataset.class_to_idx[os.path.basename(os.path.dirname(p))])
        for p, _ in rw_dataset.samples
        if os.path.basename(os.path.dirname(p)) in rw_dataset.class_to_idx
    ]
    rw_dataset.targets = [s[1] for s in rw_dataset.samples]

    rw_loader = DataLoader(rw_dataset, batch_size=BATCH_SIZE,
                           shuffle=False, num_workers=0)

    rw_preds, rw_labels = [], []
    model.eval()
    with torch.no_grad():
        for images, labels in rw_loader:
            images  = images.to(DEVICE)
            outputs = model(images)
            _, predicted = outputs.max(1)
            rw_preds.extend(predicted.cpu().numpy())
            rw_labels.extend(labels.numpy())

    rw_preds  = np.array(rw_preds)
    rw_labels = np.array(rw_labels)
    rw_acc = 100. * (rw_preds == rw_labels).sum() / len(rw_labels)

    print(f"Fine-Tuned ResNet50 — Real-World Test Accuracy: {rw_acc:.2f}%")
    print()
    print(classification_report(rw_labels, rw_preds,
                                 target_names=short_names, digits=4))

    rw_cm = confusion_matrix(rw_labels, rw_preds)
    plt.figure(figsize=(8, 7))
    sns.heatmap(rw_cm, annot=True, fmt="d", cmap="Greens",
                xticklabels=short_names, yticklabels=short_names,
                linewidths=0.5, annot_kws={"size": 11})
    plt.title("Fine-Tuned ResNet50 — Confusion Matrix (Real-World Test)",
              fontsize=13, fontweight="bold")
    plt.xlabel("Predicted"); plt.ylabel("True")
    plt.xticks(rotation=30, ha="right"); plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig("finetuned_confusion_matrix_realworld.png", dpi=150)
    plt.close()

Fine-Tuned ResNet50 — Real-World Test Accuracy: 68.75%

                               precision    recall  f1-score   support

                 Early blight     0.4375    0.7778    0.5600         9
                  Late blight     0.7778    0.7000    0.7368        10
                      healthy     0.8000    0.5000    0.6154         8
Tomato Yellow Leaf Curl Virus     0.9286    0.8667    0.8966        15
                    Leaf Mold     0.5000    0.3333    0.4000         6

                     accuracy                         0.6875        48
                    macro avg     0.6888    0.6356    0.6418        48
                 weighted avg     0.7301    0.6875    0.6912        48

